# ДЗ 3. NumPy на данных «Дала Маркет»
Срок: **до четверга 12.11, 19:00**.
Сдача — папка `hw03` репозитория `dsa-homework` (`hw03.ipynb`): `git add hw03` → `git commit` → `git push`.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

- **[База]** обязательно для всех · **[Стандарт]** — сколько получится · **[Челлендж]** — для тех, кому легко.

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])
```

Мои предсказания и объяснения:

1. 
2. 
3. 
4. 
5. 

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [ ]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

## [База]
1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [ ]:
# База
# [База]

# 1. Индекс самого дорогого заказа
idx = amount.argmax()
print('1. Самый дорогой заказ:')
print(f'   Позиция (с нуля): {idx}, сумма: {amount[idx]:,.0f} ₸, клиент: {cust[idx]}, город: {city[idx]}')
print(f'   Для менеджера: максимальный заказ — {amount[idx]:,.0f} ₸, его оформил клиент {cust[idx]} из города {city[idx]}.')

# 2. Выручка оплаченных и сумма неоплаченных заказов
paid_total = amount[paid].sum()
unpaid_total = amount[~paid].sum()
print('\n2. Суммы заказов:')
print(f'   Оплаченные: {paid_total:,.0f} ₸; неоплаченные: {unpaid_total:,.0f} ₸')
print(f'   Для менеджера: оплачено на {paid_total:,.0f} ₸, ещё {unpaid_total:,.0f} ₸ приходится на неоплаченные заказы.')

# 3. Заказы в Шымкенте
shymkent_mask = city == 'Шымкент'
shymkent_count = shymkent_mask.sum()
shymkent_share = shymkent_mask.mean() * 100
print('\n3. Шымкент:')
print(f'   Заказов: {shymkent_count}; доля: {shymkent_share:.2f}%')
print(f'   Для менеджера: {shymkent_count} заказов ({shymkent_share:.2f}% общего количества) оформлены в Шымкенте.')

## [Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [ ]:
# Стандарт
# 4. Скидка 15% только на оплаченные заказы дороже 50 000 ₸
discount_mask = paid & (amount > 50_000)
loss_all = amount[discount_mask].sum() * 0.15
loss_without_test = amount[discount_mask & (cust != 'TEST')].sum() * 0.15
print('4. Потеря выручки от скидки 15%:')
print(f'   Все клиенты: {loss_all:,.2f} ₸')
print(f'   Без TEST:    {loss_without_test:,.2f} ₸')
print(f'   Разница: в {loss_all / loss_without_test:.2f} раза')
print('   Для менеджера: TEST содержит аномально крупные тестовые заказы; без них оценка затрат на скидку реалистичнее.')

# 5. Доля доставок за 1–2 дня в каждом городе
print('\n5. Доставка за 1–2 дня по городам:')
fast_delivery = (days >= 1) & (days <= 2)
for one_city in np.unique(city):  # допустим цикл по городам, а не по заказам
    city_mask = city == one_city
    # Исключаем заказы без сведений о доставке из знаменателя
    known_days = city_mask & ~np.isnan(days)
    fast_count = (known_days & fast_delivery).sum()
    known_count = known_days.sum()
    share = fast_count / known_count * 100 if known_count else float('nan')
    print(f'   {one_city}: {fast_count} из {known_count} ({share:.2f}%)')
print('   Для менеджера: сравнение городов рассчитано по заказам с указанными сроками доставки.')

# 6. Две ошибки исходного кода:
#    1) amount.mean() считает и неоплаченные заказы;
#    2) TEST с аномально крупными заказами искажает средний чек.
# Правильно: среднее по оплаченным заказам реальных клиентов.
valid_orders = paid & (cust != 'TEST')
avg_correct = amount[valid_orders].mean()
print('\n6. Средний чек:')
print(f'   Ошибочный, по всем заказам: {amount.mean():,.2f} ₸')
print(f'   Только оплаченные (с TEST): {amount[paid].mean():,.2f} ₸')
print(f'   Исправленный, оплаченные без TEST: {avg_correct:,.2f} ₸')
print(f'   Для менеджера: средний чек реальных оплаченных заказов — {avg_correct:,.0f} ₸; TEST исключён.')

## [Челлендж]
7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
   (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
   Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [ ]:
# Челлендж